# 🛡️ Fall Detection — Cloud GPU (MediaPipe + TCN)

Clone repo + chạy backend với **model train (MediaPipe + best_model.tflite)** trên GPU.

**Trước khi chạy:** Runtime → Change runtime type → GPU (T4) → Save. Rồi Run all.

## 1) Clone repo + cài đặt (gồm mediapipe)

In [ ]:
!git clone -q https://github.com/NguyettNhu/smart_data.git || (cd smart_data && git pull -q)
%cd /content/smart_data/backend
!pip -q install -r requirements.txt python-dotenv mediapipe
# Colab fix: requirements pins pillow>=12.1 which breaks PIL._typing -> clean pillow<12
!pip -q install --force-reinstall --no-cache-dir "pillow<12"
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2) Chạy backend (FALL_ENGINE=mediapipe) + tunnel

Chờ in ra `BACKEND_PUBLIC_URL https://....trycloudflare.com`.

In [ ]:
import os, sys, re, time, threading, subprocess, uvicorn

# Use the trained MediaPipe + TCN fall engine (model files in backend/model_v8/)
os.environ["FALL_ENGINE"] = "mediapipe"
os.environ["MODEL_PATH"] = "models/fall_yolo.pt"   # YOLO still loads for /predict; unused by /ws

sys.path.insert(0, "/content/smart_data/backend")
os.chdir("/content/smart_data/backend")
from main import app   # MediaPipe + TFLite load lazily on the first frame

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_serve, daemon=True).start()
time.sleep(6)

proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
for line in proc.stdout:
    m = re.search(r"https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break
print("BACKEND_PUBLIC_URL", public_url)

## 3) Kết nối frontend
1. Copy URL.
2. frontend/.env.local: NEXT_PUBLIC_API_BASE=<URL>
3. Khởi động lại next dev.